In [1]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# inicializa conexão (MT5 já aberto manualmente)
if not mt5.initialize():
    raise RuntimeError(f"❌ Falha ao inicializar MT5: {mt5.last_error()}")

print("✅ MT5 conectado")

account = mt5.account_info()
print(account)


✅ MT5 conectado
AccountInfo(login=18901295, trade_mode=2, leverage=1, limit_orders=0, margin_so_mode=1, trade_allowed=True, trade_expert=True, margin_mode=2, currency_digits=2, fifo_close=False, balance=8937.0, credit=0.0, profit=-587.0, equity=8350.0, margin=4988.0, margin_free=3362.0, margin_level=167.401764234162, margin_so_call=0.0, margin_so_so=0.0, margin_initial=0.0, margin_maintenance=0.0, assets=0.0, liabilities=0.0, commission_blocked=0.0, name='RAFAEL LANZA CARIOCA', server='XPMT5-PRD', currency='BRL', company='XP Investimentos CCTVM S/A')


In [2]:
# ===============================
# CLASSIFICAÇÃO CANÔNICA (XP / B3)
# ===============================

def classify_type(info):
    """
    Classificação correta usando path (XP / B3):
    - Se path contém 'OPCOES' → é opção
        - option_right == 0 → CALL
        - option_right == 1 → PUT
        - senão → OPTION
    - Caso contrário → SPOT
    """
    if info is None or info.path is None:
        return "UNKNOWN"

    path = info.path.upper()

    if "OPCOES" in path:
        if info.option_right == 0:
            return "CALL"
        elif info.option_right == 1:
            return "PUT"
        else:
            return "OPTION"

    return "SPOT"


def d1_close_and_volume(ticker):

    rates = mt5.copy_rates_range(
        ticker,
        mt5.TIMEFRAME_D1,
        datetime.utcnow() - timedelta(days=15),
        datetime.utcnow()
    )

    fechamento_d1 = np.nan
    volume_d1 = np.nan

    if rates is not None and len(rates) >= 2:

        # D-1 = último candle fechado
        candle_d1 = rates[-1]

        fechamento_d1 = float(candle_d1["close"])

        if candle_d1["real_volume"] > 0:
            volume_d1 = int(candle_d1["real_volume"])
        else:
            volume_d1 = int(candle_d1["tick_volume"])

    return fechamento_d1, volume_d1

# ===============================
# MONTA LINHA DA TABELA
# ===============================

def build_row(ticker):
    mt5.symbol_select(ticker, True)
    info = mt5.symbol_info(ticker)

    description = info.description if info else None
    option_right = info.option_right if info else None

    strike = (
        float(info.option_strike)
        if info and info.option_strike and info.option_strike > 0
        else np.nan
    )

    expiration = pd.NaT
    try:
        if info and info.expiration_time and info.expiration_time > 0:
            expiration = datetime.fromtimestamp(info.expiration_time)
    except Exception:
        expiration = pd.NaT

    asset_type = classify_type(info)

    fechamento_d1, volume_d1 = d1_close_and_volume(ticker)

    path = info.path if info else None

    return {
        "ticker": ticker,
        "description": description,
        "type": asset_type,
        "option_right": option_right,
        "strike": strike,
        "expiration": expiration,
        "fechamento_d-1": fechamento_d1,
        "volume_d-1": volume_d1,
        "path": path
    }


# ===============================
# EXECUÇÃO (TESTE CONTROLADO)
# ===============================

tickers = ["PETR4", "PETRC200", "ABEV3Q"]

rows = [build_row(t) for t in tickers]

df_ativos = pd.DataFrame(rows)

df_ativos


,ticker,description,type,option_right,strike,expiration,fechamento_d-1,volume_d-1,path
0,PETR4,PETROBRAS PN ATZ N2,SPOT,0,NaN,NaT,48.1,37743000.0,BOVESPA\A VISTA\PETR4
1,PETRC200,"PETR /ATZPN 17,75",CALL,0,17.75,2027-03-19 20:59:59,34.2,100.0,BOVESPA\OPCOES\PETRC200
2,ABEV3Q,AMBEV S/A ON ATZ,SPOT,0,NaN,NaT,NaN,NaN,BOVESPA\A VISTA\ABEV3Q


In [ ]:
# ===============================
# UNDERLYINGS LÍQUIDOS (BASE)
# ===============================

UNDERLYINGS_BASE = [
    "PETR",
    "VALE",
    "ITUB",
    "BBDC",
    "BBAS",
    "ABEV",
    "WEGE",
    "B3SA",
    "SUZB",
]

# ===============================
# CLASSIFICAÇÃO CANÔNICA (XP / B3)
# ===============================

def classify_type(info):
    """
    Classificação correta usando path (XP / B3):
    - Se path contém 'OPCOES' → é opção
        - option_right == 0 → CALL
        - option_right == 1 → PUT
        - senão → OPTION
    - Caso contrário → SPOT
    """
    if info is None or info.path is None:
        return "UNKNOWN"

    path = info.path.upper()

    if "OPCOES" in path:
        if info.option_right == 0:
            return "CALL"
        elif info.option_right == 1:
            return "PUT"
        else:
            return "OPTION"

    return "SPOT"


# ===============================
# FECHAMENTO E VOLUME (MANTÉM [-1])
# ===============================

def d1_close_and_volume(ticker):
    rates = mt5.copy_rates_range(
        ticker,
        mt5.TIMEFRAME_D1,
        datetime.utcnow() - timedelta(days=15),
        datetime.utcnow()
    )

    fechamento_d1 = np.nan
    volume_d1 = np.nan

    if rates is not None and len(rates) >= 2:
        candle = rates[-1]  # <-- MANTIDO COMO DEFINIDO

        fechamento_d1 = float(candle["close"])

        if candle["real_volume"] > 0:
            volume_d1 = int(candle["real_volume"])
        else:
            volume_d1 = int(candle["tick_volume"])  # fallback

    return fechamento_d1, volume_d1


# ===============================
# MONTA LINHA DA TABELA
# ===============================

def build_row(ticker):
    mt5.symbol_select(ticker, True)
    info = mt5.symbol_info(ticker)

    description = info.description if info else None
    option_right = info.option_right if info else None

    strike = (
        float(info.option_strike)
        if info and info.option_strike and info.option_strike > 0
        else np.nan
    )

    expiration = pd.NaT
    try:
        if info and info.expiration_time and info.expiration_time > 0:
            expiration = datetime.fromtimestamp(info.expiration_time)
    except Exception:
        expiration = pd.NaT

    asset_type = classify_type(info)
    fechamento_d1, volume_d1 = d1_close_and_volume(ticker)
    path = info.path if info else None

    return {
        "ticker": ticker,
        "description": description,
        "type": asset_type,
        "option_right": option_right,
        "strike": strike,
        "expiration": expiration,
        "fechamento_d-1": fechamento_d1,
        "volume_d-1": volume_d1,
        "path": path
    }


# ===============================
# PRÓXIMO VENCIMENTO (3ª SEXTA)
# ===============================

def third_friday(year, month):
    d = datetime(year, month, 1)
    days_until_friday = (4 - d.weekday()) % 7  # Friday = 4
    first_friday = d + timedelta(days=days_until_friday)
    third = first_friday + timedelta(days=14)
    return third.date()

now = datetime.utcnow()

current_expiry_date = third_friday(now.year, now.month)

if now.month == 12:
    next_month = 1
    next_year = now.year + 1
else:
    next_month = now.month + 1
    next_year = now.year

next_expiry_date = third_friday(next_year, next_month)

# datetime alvo do vencimento (3ª sexta do próximo mês)
target_expiry = datetime(
    next_year,
    next_month,
    next_expiry_date.day
)


# ===============================
# GERAR TICKERS (SÓ LÍQUIDOS)
# ===============================

import re

# ===============================
# GERAR TICKERS (SÓ LÍQUIDOS)
# ===============================

symbols = mt5.symbols_get() or []
tickers = []

# SPOT "padrão B3": 4 letras + (3|4|11)
SPOT_RE = re.compile(r"^[A-Z]{4}(3|4|11)$")

for s in symbols:
    name = s.name.upper()
    path = getattr(s, "path", "")
    if not path:
        continue

    # só underlyings líquidos
    if not any(name.startswith(u) for u in UNDERLYINGS_BASE):
        continue

    path_up = path.upper()

    # -----------------------------
    # AÇÕES À VISTA (somente padrão)
    # -----------------------------
    if "VISTA" in path_up:
        if SPOT_RE.match(name):
            tickers.append(s.name)
        continue

    # -----------------------------
    # OPÇÕES (somente vencimento alvo)
    # -----------------------------
    if "OPCO" in path_up:
        info = mt5.symbol_info(s.name)
        if not info or not info.expiration_time or info.expiration_time <= 0:
            continue

        try:
            exp = datetime.fromtimestamp(info.expiration_time)
        except Exception:
            continue

        # filtro forte: ano/mês e DIA exato da 3ª sexta do próximo mês
        exp_date = exp.date()

        if exp_date == current_expiry_date or exp_date == next_expiry_date:
            tickers.append(s.name)

print("Qtd de tickers selecionados:", len(tickers))
print("Primeiros 10:", tickers[:10])


rows = [build_row(t) for t in tickers]
df_ativos = pd.DataFrame(rows)

df_ativos.head(10)


Qtd de tickers selecionados: 5754
Primeiros 10: ['BBAS11', 'BBAS3', 'BBDC3', 'BBDC4', 'ITUB3', 'ITUB4', 'PETR3', 'PETR4', 'SUZB3', 'VALE3']


In [ ]:
# =========================================
# ENRIQUECIMENTO: UNDERLYING (CHAVE DE CRUZAMENTO)
# =========================================

import re

def infer_underlying(row):
    ticker = row["ticker"]
    asset_type = row["type"]
    desc = row["description"]

    base = ticker[:4]

    # SPOT: o próprio ativo
    if asset_type == "SPOT":
        return ticker

    # CALL / PUT: usa ON / PN da description
    if asset_type in ("CALL", "PUT") and isinstance(desc, str):
        d = desc.upper()

        # procura ON ou PN como palavra
        if re.search(r"\bPN\b", d):
            return f"{base}4"
        if re.search(r"\bON\b", d):
            return f"{base}3"

    return None


df_ativos["underlying"] = df_ativos.apply(infer_underlying, axis=1)
df_ativos.head(10)


,ticker,description,type,option_right,strike,expiration,fechamento_d-1,volume_d-1,path,underlying
0,BBAS11,BRASIL ON SJ1 NM,SPOT,0,NaN,NaT,NaN,NaN,BOVESPA\A VISTA\BBAS11,BBAS11
1,BBAS3,BRASIL ON NM,SPOT,0,NaN,NaT,23.21,11350400.0,BOVESPA\A VISTA\BBAS3,BBAS3
2,BBDC3,BRADESCO ON N1,SPOT,0,NaN,NaT,15.89,2416800.0,BOVESPA\A VISTA\BBDC3,BBDC3
3,BBDC4,BRADESCO PN N1,SPOT,0,NaN,NaT,18.36,12380800.0,BOVESPA\A VISTA\BBDC4,BBDC4
4,ITUB3,ITAUUNIBANCOON N1,SPOT,0,NaN,NaT,40.93,799900.0,BOVESPA\A VISTA\ITUB3,ITUB3
5,ITUB4,ITAUUNIBANCOPN N1,SPOT,0,NaN,NaT,41.96,10808300.0,BOVESPA\A VISTA\ITUB4,ITUB4
6,PETR3,PETROBRAS ON N2,SPOT,0,NaN,NaT,53.09,7695000.0,BOVESPA\A VISTA\PETR3,PETR3
7,PETR4,PETROBRAS PN N2,SPOT,0,NaN,NaT,47.97,34977100.0,BOVESPA\A VISTA\PETR4,PETR4
8,SUZB3,SUZANO S.A. ON NM,SPOT,0,NaN,NaT,51.49,2498600.0,BOVESPA\A VISTA\SUZB3,SUZB3
9,VALE3,VALE ON NM,SPOT,0,NaN,NaT,76.06,14311100.0,BOVESPA\A VISTA\VALE3,VALE3


In [ ]:
# =========================================
# 1) UNIVERSO: SPOT + CALLs
# =========================================

df_spot = df_ativos[df_ativos["type"] == "SPOT"].copy()
df_calls = df_ativos[df_ativos["type"] == "CALL"].copy()


In [ ]:
# =========================================
# 2) PREÇO DO SPOT NAS CALLs
# =========================================

spot_price_map = dict(
    zip(df_spot["ticker"], df_spot["fechamento_d-1"])
)

df_calls["spot_price"] = df_calls["underlying"].map(spot_price_map)
df_calls.head(10)



,ticker,description,type,option_right,strike,expiration,fechamento_d-1,volume_d-1,path,underlying,spot_price
11,PETRC276,"PETR PN 26,15",CALL,0,26.15,2026-03-20 20:59:59,21.84,7500.0,BOVESPA\OPCOES\PETRC276,PETR4,47.97
12,PETRD301,"PETRE PN 27,90",CALL,0,27.90,2026-04-17 20:59:59,19.39,4200.0,BOVESPA\OPCOES\PETRD301,PETR4,47.97
15,VALEC500,"VALE ON 46,68",CALL,0,46.68,2026-03-20 20:59:59,31.25,900.0,BOVESPA\OPCOES\VALEC500,VALE3,76.06
17,PETRC261,"PETR PN 24,65",CALL,0,24.65,2026-03-20 20:59:59,21.00,1100.0,BOVESPA\OPCOES\PETRC261,PETR4,47.97
18,PETRD291,"PETR PN 27,65",CALL,0,27.65,2026-04-17 20:59:59,19.53,100.0,BOVESPA\OPCOES\PETRD291,PETR4,47.97
20,PETRD276,"PETR PN 26,15",CALL,0,26.15,2026-04-17 20:59:59,21.90,4100.0,BOVESPA\OPCOES\PETRD276,PETR4,47.97
21,ITUBC339,"ITUBE PN 30,72",CALL,0,30.72,2026-03-20 20:59:59,11.50,400.0,BOVESPA\OPCOES\ITUBC339,ITUB4,41.96
22,BBASD347,"BBAS ON 34,52",CALL,0,34.52,2026-04-17 20:59:59,NaN,NaN,BOVESPA\OPCOES\BBASD347,BBAS3,23.21
24,ITUBC264,"ITUBE PN 23,44",CALL,0,23.44,2026-03-20 20:59:59,NaN,NaN,BOVESPA\OPCOES\ITUBC264,ITUB4,41.96
25,ITUBD365,"ITUB PN 36,55",CALL,0,36.55,2026-04-17 20:59:59,6.39,100.0,BOVESPA\OPCOES\ITUBD365,ITUB4,41.96


In [ ]:
# =========================================
# 3) CALLs OTM (candidatas a crédito)
# =========================================

df_calls_valid = df_calls[
    (df_calls["strike"].notna()) &
    (df_calls["spot_price"].notna()) &
    (df_calls["strike"] > df_calls["spot_price"])
].copy()


In [ ]:
# =========================================
# 4) GERAR CALL CREDIT SPREADS (COM VOLUME)
# =========================================

spreads = []

group_cols = ["underlying", "expiration"]

for (underlying, expiration), g in df_calls_valid.groupby(group_cols):
    g = g.sort_values("strike")

    for _, sell in g.iterrows():
        for _, buy in g.iterrows():
            if buy["strike"] <= sell["strike"]:
                continue

            credit = sell["fechamento_d-1"] - buy["fechamento_d-1"]

            if credit <= 0:
                continue

            spreads.append({
                "underlying": underlying,
                "expiration": expiration,

                "sell_ticker": sell["ticker"],
                "buy_ticker": buy["ticker"],

                "sell_strike": sell["strike"],
                "buy_strike": buy["strike"],

                "spot_price": sell["spot_price"],

                "sell_premium": sell["fechamento_d-1"],
                "buy_premium": buy["fechamento_d-1"],
                "credit": credit,

                # 👇 volumes individuais das pernas
                "sell_volume": sell["volume_d-1"],
                "buy_volume": buy["volume_d-1"],
            })

df_call_spreads = pd.DataFrame(spreads)
df_call_spreads.head(10)


,underlying,expiration,sell_ticker,buy_ticker,sell_strike,buy_strike,spot_price,sell_premium,buy_premium,credit,sell_volume,buy_volume
0,ABEV3,2026-03-20 20:59:59,ABEVC153,ABEVC156,14.69,14.94,14.59,0.1,0.03,0.07,264600.0,61100.0
1,ABEV3,2026-03-20 20:59:59,ABEVC153,ABEVC158,14.69,15.19,14.59,0.1,0.01,0.09,264600.0,93700.0
2,ABEV3,2026-03-20 20:59:59,ABEVC153,ABEVC161,14.69,15.44,14.59,0.1,0.01,0.09,264600.0,13300.0
3,ABEV3,2026-03-20 20:59:59,ABEVC153,ABEVC163,14.69,15.69,14.59,0.1,0.02,0.08,264600.0,12500.0
4,ABEV3,2026-03-20 20:59:59,ABEVC153,ABEVC166,14.69,15.94,14.59,0.1,0.01,0.09,264600.0,55200.0
5,ABEV3,2026-03-20 20:59:59,ABEVC153,ABEVC168,14.69,16.19,14.59,0.1,0.01,0.09,264600.0,6900.0
6,ABEV3,2026-03-20 20:59:59,ABEVC153,ABEVC171,14.69,16.44,14.59,0.1,0.01,0.09,264600.0,12200.0
7,ABEV3,2026-03-20 20:59:59,ABEVC153,ABEVC173,14.69,16.69,14.59,0.1,0.01,0.09,264600.0,300.0
8,ABEV3,2026-03-20 20:59:59,ABEVC153,ABEVC176,14.69,16.94,14.59,0.1,0.01,0.09,264600.0,32500.0
9,ABEV3,2026-03-20 20:59:59,ABEVC153,ABEVC178,14.69,17.19,14.59,0.1,0.01,0.09,264600.0,2500.0


In [ ]:
# =========================================
# BLOCO FINAL — FILTROS + MÉTRICAS + CONTAGEM
# =========================================

# -------------------------------
# PARÂMETROS
# -------------------------------
MIN_SELL_VOLUME    = 5000
MIN_BUY_VOLUME     = 1000
MIN_OTM_PCT        = 2.0
MIN_RETURN_PCT     = 30.0
MIN_CREDIT         = 0.5

MIN_SPREAD_WIDTH   = 1.0
MAX_SPREAD_WIDTH   = 2.0


# -------------------------------
# MÉTRICAS
# -------------------------------
df = df_call_spreads.copy()

df["spread_width"] = df["buy_strike"] - df["sell_strike"]

df["otm_pct"] = (
    (df["sell_strike"] - df["spot_price"]) / df["spot_price"]
) * 100

df["max_loss"] = df["spread_width"] - df["credit"]

df["return_pct"] = (df["credit"] / df["max_loss"]) * 100


# -------------------------------
# FILTROS
# -------------------------------
df_filt = df[
    (df["sell_volume"]   >= MIN_SELL_VOLUME)   &
    (df["buy_volume"]    >= MIN_BUY_VOLUME)    &
    (df["otm_pct"]       >= MIN_OTM_PCT)       &
    (df["return_pct"]    >= MIN_RETURN_PCT)    &
    (df["credit"]        >= MIN_CREDIT)        &
    (df["spread_width"]  >= MIN_SPREAD_WIDTH)  &
    (df["spread_width"]  <= MAX_SPREAD_WIDTH)  &
    (df["max_loss"]      > 0)
].copy()


# =========================================
# SEPARA VENCIMENTOS
# =========================================

df_filt["exp_date"] = df_filt["expiration"].dt.date

df_current = df_filt[df_filt["exp_date"] == current_expiry_date]
df_next    = df_filt[df_filt["exp_date"] == next_expiry_date]


# =========================================
# FUNÇÃO PARA EXIBIR TABELA
# =========================================

def show_table(df_input, titulo):

    print("")
    print("================================")
    print(titulo)
    print("================================")

    if df_input.empty:
        print("Nenhuma trava encontrada")
        return

    df_rank = df_input.sort_values(
        ["return_pct","otm_pct"],
        ascending=[False, False]
    )

    display(df_rank.head(20))


# =========================================
# TABELAS
# =========================================

show_table(df_current, "TRAVAS — VENCIMENTO MÊS ATUAL")

show_table(df_next, "TRAVAS — VENCIMENTO PRÓXIMO MÊS")


TRAVAS — VENCIMENTO MÊS ATUAL
Nenhuma trava encontrada

TRAVAS — VENCIMENTO PRÓXIMO MÊS


,underlying,expiration,sell_ticker,buy_ticker,sell_strike,buy_strike,spot_price,sell_premium,buy_premium,credit,sell_volume,buy_volume,spread_width,otm_pct,max_loss,return_pct,exp_date
39015,WEGE3,2026-04-17 20:59:59,WEGED478,WEGED48,47.86,48.86,45.72,1.24,0.63,0.61,7600.0,2200.0,1.00,4.680665,0.39,156.410256,2026-04-17
26449,VALE3,2026-04-17 20:59:59,VALED779,VALED789,77.90,78.90,76.06,2.37,1.84,0.53,82100.0,429300.0,1.00,2.419143,0.47,112.765957,2026-04-17
26450,VALE3,2026-04-17 20:59:59,VALED779,VALED794,77.90,79.40,76.06,2.37,1.66,0.71,82100.0,416200.0,1.50,2.419143,0.79,89.873418,2026-04-17
39017,WEGE3,2026-04-17 20:59:59,WEGED478,WEGED492,47.86,49.36,45.72,1.24,0.58,0.66,7600.0,3000.0,1.50,4.680665,0.84,78.571429,2026-04-17
38923,WEGE3,2026-04-17 20:59:59,WEGED473,WEGED48,47.36,48.86,45.72,1.25,0.63,0.62,13500.0,2200.0,1.50,3.587052,0.88,70.454545,2026-04-17
26451,VALE3,2026-04-17 20:59:59,VALED779,VALED799,77.90,79.90,76.06,2.37,1.57,0.80,82100.0,295400.0,2.00,2.419143,1.20,66.666667,2026-04-17
17414,SUZB3,2026-04-17 20:59:59,SUZBD540,SUZBD555,52.88,54.38,51.49,1.44,0.85,0.59,7200.0,7000.0,1.50,2.699553,0.91,64.835165,2026-04-17
39019,WEGE3,2026-04-17 20:59:59,WEGED478,WEGED49,47.86,49.86,45.72,1.24,0.50,0.74,7600.0,17800.0,2.00,4.680665,1.26,58.730159,2026-04-17
38828,WEGE3,2026-04-17 20:59:59,WEGED468,WEGED48,46.86,48.86,45.72,1.36,0.63,0.73,9300.0,2200.0,2.00,2.493438,1.27,57.480315,2026-04-17
26603,VALE3,2026-04-17 20:59:59,VALED784,VALED804,78.40,80.40,76.06,2.07,1.36,0.71,521400.0,561000.0,2.00,3.076519,1.29,55.038760,2026-04-17
